# Notebook 126: Attention ― Transformerへの架け橋
## Attention Mechanism: Bridge to the Transformer
---
### このノートブックの位置づけ
**Unit 8「シーケンスモデリング」** の最終章。NB125で実証したSeq2Seqのボトルネック問題をAttentionで解決し、Transformerへの自然な橋渡しを行います。

### 学習目標
1. **Bahdanau（加算的）Attention** を実装し、ボトルネック問題の解決を確認できる
2. **Attentionスコアマトリクス** を可視化し、「何に注目しているか」を解釈できる
3. **Scaled Dot-Product Attention** の数式を理解し、実装できる
4. **Self-Attention** の概念を理解し、Transformerへの接続を説明できる

### 前提知識
- Notebook 125（Seq2Seq、コンテキストベクトル、ボトルネック問題）
- PyTorchの基礎

⏱️ **推定学習時間**: 150-180分  
📊 **難易度**: ★★★★☆（上級）  
🎓 **カテゴリ**: 応用

## 目次

1. [Seq2Seqのボトルネック問題（復習）](#1.-Seq2Seqのボトルネック問題（復習）)
2. [Attention: 動的なコンテキスト](#2.-Attention:-動的なコンテキスト)
3. [Bahdanau (Additive) Attention](#3.-Bahdanau-(Additive)-Attention)
4. [Attention付きSeq2Seq実装](#4.-Attention付きSeq2Seq実装)
5. [Attentionマトリクスの可視化](#5.-Attentionマトリクスの可視化)
6. [Scaled Dot-Product Attention](#6.-Scaled-Dot-Product-Attention)
7. [Self-Attentionへの布石](#7.-Self-Attentionへの布石)
8. [Unit 8 総まとめ](#8.-Unit-8-総まとめ)
9. [自己評価クイズ](#9.-自己評価クイズ)

In [ ]:
# ============================================================
# セットアップ
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = ['Hiragino Sans', 'Arial Unicode MS', 'sans-serif']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

np.random.seed(42)
torch.manual_seed(42)
device = torch.device('cpu')
print(f"PyTorch version: {torch.__version__}")
print("環境セットアップ完了")

# ============================================================
# 1. Seq2Seqのボトルネック問題（復習）
# ============================================================

## なぜAttentionが必要なのか？

NB125で学んだSeq2Seqモデルには致命的な弱点がありました。

### ボトルネック問題のおさらい

- **コンテキストベクトル**: 入力系列全体を**たった1つの固定長ベクトル**に圧縮
- **問題**: 入力が長くなるほど、1つのベクトルに詰め込む情報量が増大
- **結果**: 長い系列では初期の情報が失われ、性能が劣化

### 解決のアイデア

「Decoderが最後のEncoder隠れ状態だけでなく、**すべてのEncoder隠れ状態を参照**できたらどうか？」

→ これが **Attention** メカニズムの核心です。

In [ ]:
# ============================================================
# ボトルネック問題の可視化: Seq2Seq vs Attention
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- 左: 従来のSeq2Seq ---
ax = axes[0]
ax.set_xlim(-1, 11)
ax.set_ylim(-1, 7)
ax.set_title('従来のSeq2Seq: ボトルネック', fontsize=14, fontweight='bold')
ax.axis('off')

# Encoder boxes
encoder_labels = ['h₁', 'h₂', 'h₃', 'h₄', 'h₅']
for i, label in enumerate(encoder_labels):
    rect = plt.Rectangle((i*1.8, 4), 1.2, 1, fill=True, 
                          facecolor='#3498db', edgecolor='black', alpha=0.8)
    ax.add_patch(rect)
    ax.text(i*1.8+0.6, 4.5, label, ha='center', va='center', fontsize=12, color='white', fontweight='bold')

# Context vector (bottleneck)
rect = plt.Rectangle((3.6, 2), 1.5, 1, fill=True,
                      facecolor='#e74c3c', edgecolor='black', alpha=0.9, linewidth=2)
ax.add_patch(rect)
ax.text(4.35, 2.5, 'c', ha='center', va='center', fontsize=14, color='white', fontweight='bold')
ax.text(4.35, 1.5, '← ボトルネック!', ha='center', va='center', fontsize=11, color='#e74c3c', fontweight='bold')

# Only last encoder connects to context
ax.annotate('', xy=(4.35, 3.0), xytext=(7.8, 4.0),
            arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=3))

# Decoder boxes
decoder_labels = ['s₁', 's₂', 's₃']
for i, label in enumerate(decoder_labels):
    rect = plt.Rectangle((2+i*2.5, 0), 1.2, 1, fill=True,
                          facecolor='#2ecc71', edgecolor='black', alpha=0.8)
    ax.add_patch(rect)
    ax.text(2+i*2.5+0.6, 0.5, label, ha='center', va='center', fontsize=12, color='white', fontweight='bold')

# Context to all decoders
for i in range(3):
    ax.annotate('', xy=(2+i*2.5+0.6, 1.0), xytext=(4.35, 2.0),
                arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=1.5, ls='--'))

ax.text(5, 6, 'Encoder', fontsize=12, fontweight='bold', color='#3498db')
ax.text(5, -0.5, 'Decoder', fontsize=12, fontweight='bold', color='#2ecc71')

# --- 右: Attention ---
ax = axes[1]
ax.set_xlim(-1, 11)
ax.set_ylim(-1, 7)
ax.set_title('Attention: すべての隠れ状態を参照', fontsize=14, fontweight='bold')
ax.axis('off')

# Encoder boxes
for i, label in enumerate(encoder_labels):
    rect = plt.Rectangle((i*1.8, 4), 1.2, 1, fill=True,
                          facecolor='#3498db', edgecolor='black', alpha=0.8)
    ax.add_patch(rect)
    ax.text(i*1.8+0.6, 4.5, label, ha='center', va='center', fontsize=12, color='white', fontweight='bold')

# Decoder boxes
for i, label in enumerate(decoder_labels):
    rect = plt.Rectangle((2+i*2.5, 0), 1.2, 1, fill=True,
                          facecolor='#2ecc71', edgecolor='black', alpha=0.8)
    ax.add_patch(rect)
    ax.text(2+i*2.5+0.6, 0.5, label, ha='center', va='center', fontsize=12, color='white', fontweight='bold')

# Attention connections (many-to-many)
colors = ['#e74c3c', '#f39c12', '#9b59b6']
alphas_list = [
    [0.1, 0.15, 0.2, 0.25, 0.8],  # s1 attends mostly to h5
    [0.2, 0.15, 0.7, 0.1, 0.05],   # s2 attends mostly to h3
    [0.8, 0.1, 0.05, 0.03, 0.02],  # s3 attends mostly to h1
]
for i in range(3):
    for j in range(5):
        alpha = alphas_list[i][j]
        ax.annotate('', xy=(2+i*2.5+0.6, 1.0), xytext=(j*1.8+0.6, 4.0),
                    arrowprops=dict(arrowstyle='->', color=colors[i], 
                                   lw=alpha*5, alpha=max(alpha, 0.15)))

ax.text(5, 6, 'Encoder', fontsize=12, fontweight='bold', color='#3498db')
ax.text(5, -0.5, 'Decoder（動的に注目先を変更）', fontsize=12, fontweight='bold', color='#2ecc71')

plt.suptitle('Seq2Seq vs Attention: 情報の流れの違い', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# ============================================================
# 2. Attention: 動的なコンテキスト
# ============================================================

## Attentionの核心的アイデア

Attentionメカニズムの基本的な考え方は非常にシンプルです。

### 従来のSeq2Seq
- Decoderの各ステップで、**同じ固定コンテキストベクトル**を使う

### Attention付きSeq2Seq  
- Decoderの各ステップで、**動的に計算された異なるコンテキスト**を使う
- 「今この出力を生成するのに、入力のどこが最も重要か？」を毎回判断

### 直感的な理解

外国語の文を翻訳することを想像してください：
- 翻訳する単語ごとに、元の文の**異なる部分**に目を向けますよね
- 「I love cats」を「猫が好きです」に翻訳するとき：
  - 「猫」を生成 → "cats" に注目
  - 「好き」を生成 → "love" に注目
  - 「です」を生成 → 文全体のニュアンスに注目

### Attentionの数式

Decoderの各タイムステップ $t$ で：

1. **スコア計算**: $\text{score}(s_t, h_i)$ — Decoder状態 $s_t$ に対するEncoder状態 $h_i$ の関連度

2. **重み正規化**: $\alpha_{ti} = \text{softmax}(\text{score}(s_t, h_i))$ — Attention重み

3. **コンテキスト計算**: $c_t = \sum_i \alpha_{ti} \cdot h_i$ — 重み付きコンテキスト

ポイントは、スコア関数の設計にいくつかのバリエーションがあることです。

# ============================================================
# 3. Bahdanau (Additive) Attention
# ============================================================

## Bahdanau et al. (2015) — 加算的Attention

最初に提案されたAttentionメカニズムの一つで、**加算的（additive）Attention** と呼ばれます。

### スコア関数

$$\text{score}(s_t, h_i) = v^T \cdot \tanh(W_s \cdot s_t + W_h \cdot h_i)$$

ここで：
- $W_s$ : Decoder状態を変換する重み行列
- $W_h$ : Encoder状態を変換する重み行列  
- $v$ : 最終的にスカラースコアに変換するベクトル
- すべて**学習可能なパラメータ**

### なぜ「加算的」？

$W_s \cdot s_t$ と $W_h \cdot h_i$ を**足し算**してからtanhを通すため、「加算的」と呼ばれます。  
後で紹介するDot-Product Attentionとの対比で理解しましょう。

In [ ]:
# ============================================================
# Bahdanau Attention モジュール
# ============================================================

class BahdanauAttention(nn.Module):
    """Bahdanau (Additive) Attention
    
    score(s_t, h_i) = v^T * tanh(W_s * s_t + W_h * h_i)
    
    Parameters
    ----------
    hidden_dim : int
        隠れ状態の次元数
    """
    def __init__(self, hidden_dim):
        super().__init__()
        self.Ws = nn.Linear(hidden_dim, hidden_dim, bias=False)  # Decoder状態の変換
        self.Wh = nn.Linear(hidden_dim, hidden_dim, bias=False)  # Encoder状態の変換
        self.v = nn.Linear(hidden_dim, 1, bias=False)             # スカラースコアへ
    
    def forward(self, decoder_state, encoder_outputs):
        """
        Parameters
        ----------
        decoder_state : (batch, hidden_dim)
            現在のDecoder隠れ状態
        encoder_outputs : (batch, src_len, hidden_dim)
            全Encoder隠れ状態
        
        Returns
        -------
        context : (batch, hidden_dim)
            重み付きコンテキストベクトル
        attention_weights : (batch, src_len)
            Attention重み（合計1）
        """
        src_len = encoder_outputs.shape[1]
        
        # Decoder状態を変換し、src_len分だけ複製
        # (batch, hidden_dim) -> (batch, 1, hidden_dim) -> (batch, src_len, hidden_dim)
        s = self.Ws(decoder_state).unsqueeze(1).expand(-1, src_len, -1)
        
        # Encoder出力を変換
        # (batch, src_len, hidden_dim) -> (batch, src_len, hidden_dim)
        h = self.Wh(encoder_outputs)
        
        # 加算 + tanh でエネルギー計算
        energy = torch.tanh(s + h)  # (batch, src_len, hidden_dim)
        
        # スカラースコアに変換
        attention_scores = self.v(energy).squeeze(-1)  # (batch, src_len)
        
        # Softmaxで正規化 → Attention重み
        attention_weights = F.softmax(attention_scores, dim=1)  # (batch, src_len)
        
        # 重み付き和でコンテキスト計算
        # (batch, 1, src_len) @ (batch, src_len, hidden_dim) -> (batch, 1, hidden_dim)
        context = torch.bmm(attention_weights.unsqueeze(1), encoder_outputs).squeeze(1)
        
        return context, attention_weights


# --- 動作テスト ---
batch_size, src_len, hidden_dim = 2, 5, 8
attn = BahdanauAttention(hidden_dim)
decoder_state = torch.randn(batch_size, hidden_dim)
encoder_outputs = torch.randn(batch_size, src_len, hidden_dim)

context, weights = attn(decoder_state, encoder_outputs)
print(f"Decoder state shape: {decoder_state.shape}")
print(f"Encoder outputs shape: {encoder_outputs.shape}")
print(f"Context vector shape: {context.shape}")
print(f"Attention weights shape: {weights.shape}")
print(f"Attention weights (sample 0): {weights[0].detach().numpy().round(3)}")
print(f"Attention weights sum: {weights[0].sum().item():.4f}")
print("\n✅ Bahdanau Attention モジュール動作確認完了")

# ============================================================
# 4. Attention付きSeq2Seq実装
# ============================================================

NB125と同じ「系列反転タスク」を使い、**Attentionの効果**を実験的に確認します。

### タスク
- 入力: `[3, 1, 4, 1, 5]`
- 出力: `[5, 1, 4, 1, 3]`（反転）

### 比較対象
1. **バニラSeq2Seq**（Attentionなし）: NB125と同じ構造
2. **Attention付きSeq2Seq**: Bahdanau Attentionを追加

In [ ]:
# ============================================================
# Encoder（Seq2SeqとAttention付きで共通）
# ============================================================

class Encoder(nn.Module):
    """LSTM Encoder
    
    入力系列をすべて処理し、各タイムステップの隠れ状態を返す。
    """
    def __init__(self, vocab_size, embed_dim, hidden_dim, n_layers=1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.LSTM(embed_dim, hidden_dim, n_layers, batch_first=True)
    
    def forward(self, src):
        """
        Parameters
        ----------
        src : (batch, src_len)  入力トークンID
        
        Returns
        -------
        outputs : (batch, src_len, hidden_dim)  全タイムステップの隠れ状態
        hidden : (n_layers, batch, hidden_dim)  最終隠れ状態
        cell : (n_layers, batch, hidden_dim)  最終セル状態
        """
        embedded = self.embedding(src)  # (batch, src_len, embed_dim)
        outputs, (hidden, cell) = self.rnn(embedded)
        return outputs, hidden, cell

print("✅ Encoder 定義完了")

In [ ]:
# ============================================================
# Attention付きDecoder
# ============================================================

class AttentionDecoder(nn.Module):
    """Attention付きLSTM Decoder
    
    各デコーディングステップで、Bahdanau Attentionにより
    動的なコンテキストベクトルを計算し、入力に結合する。
    """
    def __init__(self, output_dim, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, embed_dim)
        self.attention = BahdanauAttention(hidden_dim)
        # RNNの入力 = 埋め込み + コンテキストベクトル
        self.rnn = nn.LSTM(embed_dim + hidden_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, input_token, hidden, cell, encoder_outputs):
        """
        1ステップ分のデコーディング
        
        Parameters
        ----------
        input_token : (batch,)  前ステップの出力トークン
        hidden : (1, batch, hidden_dim)  Decoder隠れ状態
        cell : (1, batch, hidden_dim)  Decoderセル状態
        encoder_outputs : (batch, src_len, hidden_dim)  全Encoder隠れ状態
        
        Returns
        -------
        prediction : (batch, output_dim)  次トークンの予測
        hidden, cell : 更新された隠れ/セル状態
        attn_weights : (batch, src_len)  Attention重み
        """
        # 埋め込み: (batch,) -> (batch, 1, embed_dim)
        embedded = self.embedding(input_token.unsqueeze(1))
        
        # Attention計算
        # hidden[-1]: 最後のレイヤーの隠れ状態 (batch, hidden_dim)
        context, attn_weights = self.attention(hidden[-1], encoder_outputs)
        
        # コンテキストと埋め込みを結合
        # (batch, 1, embed_dim + hidden_dim)
        rnn_input = torch.cat([embedded, context.unsqueeze(1)], dim=2)
        
        # RNNに通す
        output, (hidden, cell) = self.rnn(rnn_input, (hidden, cell))
        
        # 出力層
        prediction = self.fc(output.squeeze(1))  # (batch, output_dim)
        
        return prediction, hidden, cell, attn_weights

print("✅ AttentionDecoder 定義完了")

In [ ]:
# ============================================================
# Seq2Seqモデル（Attentionあり／なし）
# ============================================================

class VanillaDecoder(nn.Module):
    """Attentionなしの従来のDecoder（比較用）"""
    def __init__(self, output_dim, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, embed_dim)
        self.rnn = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, input_token, hidden, cell, encoder_outputs=None):
        embedded = self.embedding(input_token.unsqueeze(1))
        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))
        prediction = self.fc(output.squeeze(1))
        return prediction, hidden, cell, None


class Seq2Seq(nn.Module):
    """Seq2Seqモデル（Attention付き/なし 両対応）"""
    def __init__(self, encoder, decoder, sos_token, use_attention=True):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.sos_token = sos_token
        self.use_attention = use_attention
    
    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        """
        Parameters
        ----------
        src : (batch, src_len)
        trg : (batch, trg_len)
        teacher_forcing_ratio : float
        
        Returns
        -------
        outputs : (batch, trg_len, output_dim)
        all_attn_weights : list of (batch, src_len) or None
        """
        batch_size = src.shape[0]
        trg_len = trg.shape[1]
        output_dim = self.decoder.fc.out_features
        
        # Encoder
        encoder_outputs, hidden, cell = self.encoder(src)
        
        # 出力を蓄積するテンソル
        outputs = torch.zeros(batch_size, trg_len, output_dim)
        all_attn_weights = []
        
        # 最初の入力はSOSトークン
        input_token = torch.full((batch_size,), self.sos_token, dtype=torch.long)
        
        for t in range(trg_len):
            prediction, hidden, cell, attn_weights = self.decoder(
                input_token, hidden, cell, encoder_outputs
            )
            outputs[:, t, :] = prediction
            if attn_weights is not None:
                all_attn_weights.append(attn_weights.detach())
            
            # Teacher Forcing
            use_teacher = np.random.random() < teacher_forcing_ratio
            if use_teacher and t < trg_len - 1:
                input_token = trg[:, t]
            else:
                input_token = prediction.argmax(dim=1)
        
        if all_attn_weights:
            all_attn_weights = torch.stack(all_attn_weights, dim=1)  # (batch, trg_len, src_len)
        else:
            all_attn_weights = None
        
        return outputs, all_attn_weights

print("✅ Seq2Seq モデル定義完了")

In [ ]:
# ============================================================
# データ生成: 系列反転タスク
# ============================================================

def generate_reverse_data(n_samples, seq_len, vocab_size, sos_token, eos_token):
    """系列反転タスクのデータを生成
    
    入力: [3, 1, 4, 1, 5]
    出力: [5, 1, 4, 1, 3] (反転)
    
    特殊トークン（SOS, EOS）はデータに含めない（vocab_size未満の数値のみ使用）
    """
    # 特殊トークンを除いた範囲でランダム生成
    usable_vocab = vocab_size - 2  # SOS, EOS を除く
    src_data = np.random.randint(0, usable_vocab, size=(n_samples, seq_len))
    trg_data = src_data[:, ::-1].copy()  # 反転
    
    return torch.tensor(src_data, dtype=torch.long), torch.tensor(trg_data, dtype=torch.long)


# --- ハイパーパラメータ ---
VOCAB_SIZE = 12     # 0-9 の数字 + SOS(10) + EOS(11)
SOS_TOKEN = 10
EOS_TOKEN = 11
EMBED_DIM = 16
HIDDEN_DIM = 32
N_SAMPLES = 2000
BATCH_SIZE = 64
N_EPOCHS = 60
LR = 0.003

# --- さまざまな系列長でデータ生成 ---
SEQ_LENGTHS = [6, 10, 15]  # 短い、中くらい、長い

datasets = {}
for sl in SEQ_LENGTHS:
    src, trg = generate_reverse_data(N_SAMPLES, sl, VOCAB_SIZE, SOS_TOKEN, EOS_TOKEN)
    datasets[sl] = (src, trg)
    print(f"系列長 {sl}: src shape={src.shape}, trg shape={trg.shape}")
    print(f"  例: {src[0].tolist()} → {trg[0].tolist()}")

print(f"\n✅ データ生成完了（系列長: {SEQ_LENGTHS}）")

In [ ]:
# ============================================================
# 訓練ループ
# ============================================================

def train_model(model, src_data, trg_data, n_epochs, batch_size, lr, 
                teacher_forcing_ratio=0.5, verbose=True):
    """モデルを訓練し、エポックごとの損失と精度を記録"""
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    n_samples = src_data.shape[0]
    n_batches = n_samples // batch_size
    
    history = {'loss': [], 'accuracy': []}
    
    for epoch in range(n_epochs):
        # シャッフル
        perm = torch.randperm(n_samples)
        src_shuffled = src_data[perm]
        trg_shuffled = trg_data[perm]
        
        epoch_loss = 0
        epoch_correct = 0
        epoch_total = 0
        
        model.train()
        for i in range(n_batches):
            start = i * batch_size
            end = start + batch_size
            src_batch = src_shuffled[start:end]
            trg_batch = trg_shuffled[start:end]
            
            optimizer.zero_grad()
            output, _ = model(src_batch, trg_batch, teacher_forcing_ratio)
            
            # Loss: (batch * trg_len, vocab_size) vs (batch * trg_len)
            output_flat = output.reshape(-1, output.shape[-1])
            trg_flat = trg_batch.reshape(-1)
            loss = criterion(output_flat, trg_flat)
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            epoch_loss += loss.item()
            predictions = output.argmax(dim=2)
            epoch_correct += (predictions == trg_batch).sum().item()
            epoch_total += trg_batch.numel()
        
        avg_loss = epoch_loss / n_batches
        accuracy = epoch_correct / epoch_total
        history['loss'].append(avg_loss)
        history['accuracy'].append(accuracy)
        
        if verbose and (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1:3d}/{n_epochs} | Loss: {avg_loss:.4f} | Acc: {accuracy:.4f}")
    
    return history


print("✅ 訓練関数定義完了")

In [ ]:
# ============================================================
# 訓練実行: Attention vs Vanilla Seq2Seq
# ============================================================

results = {}

for seq_len in SEQ_LENGTHS:
    print(f"\n{'='*60}")
    print(f"系列長: {seq_len}")
    print(f"{'='*60}")
    
    src_data, trg_data = datasets[seq_len]
    
    # --- Vanilla Seq2Seq ---
    torch.manual_seed(42)
    enc_vanilla = Encoder(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM)
    dec_vanilla = VanillaDecoder(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM)
    model_vanilla = Seq2Seq(enc_vanilla, dec_vanilla, SOS_TOKEN, use_attention=False)
    
    print(f"\n--- Vanilla Seq2Seq ---")
    hist_vanilla = train_model(model_vanilla, src_data, trg_data, 
                                N_EPOCHS, BATCH_SIZE, LR)
    
    # --- Attention Seq2Seq ---
    torch.manual_seed(42)
    enc_attn = Encoder(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM)
    dec_attn = AttentionDecoder(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM)
    model_attn = Seq2Seq(enc_attn, dec_attn, SOS_TOKEN, use_attention=True)
    
    print(f"\n--- Attention Seq2Seq ---")
    hist_attn = train_model(model_attn, src_data, trg_data,
                             N_EPOCHS, BATCH_SIZE, LR)
    
    results[seq_len] = {
        'vanilla': hist_vanilla,
        'attention': hist_attn,
        'model_attn': model_attn,
        'model_vanilla': model_vanilla
    }

print("\n✅ 全モデルの訓練完了")

In [ ]:
# ============================================================
# 訓練曲線の比較: Attention vs Vanilla Seq2Seq
# ============================================================

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for idx, seq_len in enumerate(SEQ_LENGTHS):
    hist_v = results[seq_len]['vanilla']
    hist_a = results[seq_len]['attention']
    
    # Loss
    ax = axes[0, idx]
    ax.plot(hist_v['loss'], label='Vanilla Seq2Seq', color='#e74c3c', alpha=0.8, linewidth=2)
    ax.plot(hist_a['loss'], label='Attention Seq2Seq', color='#3498db', alpha=0.8, linewidth=2)
    ax.set_title(f'系列長 {seq_len}: 損失', fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Accuracy
    ax = axes[1, idx]
    ax.plot(hist_v['accuracy'], label='Vanilla Seq2Seq', color='#e74c3c', alpha=0.8, linewidth=2)
    ax.plot(hist_a['accuracy'], label='Attention Seq2Seq', color='#3498db', alpha=0.8, linewidth=2)
    ax.set_title(f'系列長 {seq_len}: 精度', fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 1.05)

plt.suptitle('Attention vs Vanilla Seq2Seq: 系列長による性能比較', 
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# --- 最終精度まとめ ---
print("\n最終精度の比較:")
print(f"{'系列長':>8} | {'Vanilla':>12} | {'Attention':>12} | {'差分':>10}")
print("-" * 50)
for seq_len in SEQ_LENGTHS:
    v_acc = results[seq_len]['vanilla']['accuracy'][-1]
    a_acc = results[seq_len]['attention']['accuracy'][-1]
    diff = a_acc - v_acc
    print(f"{seq_len:>8} | {v_acc:>12.4f} | {a_acc:>12.4f} | {diff:>+10.4f}")

In [ ]:
# ============================================================
# 評価: 具体的な予測例
# ============================================================

def evaluate_model(model, src_data, trg_data, n_examples=5):
    """モデルの予測を表示"""
    model.eval()
    with torch.no_grad():
        output, attn_weights = model(src_data[:n_examples], trg_data[:n_examples], 
                                      teacher_forcing_ratio=0.0)
        predictions = output.argmax(dim=2)
    
    for i in range(n_examples):
        src = src_data[i].tolist()
        trg = trg_data[i].tolist()
        pred = predictions[i].tolist()
        correct = "✅" if pred == trg else "❌"
        print(f"  入力: {src}")
        print(f"  正解: {trg}")
        print(f"  予測: {pred} {correct}")
        print()
    
    return output, attn_weights, predictions

# 系列長10で比較
test_sl = 10
src_test, trg_test = datasets[test_sl]

print(f"=== Vanilla Seq2Seq (系列長{test_sl}) ===")
_, _, _ = evaluate_model(results[test_sl]['model_vanilla'], src_test, trg_test)

print(f"=== Attention Seq2Seq (系列長{test_sl}) ===")
output_attn, attn_wts, preds_attn = evaluate_model(results[test_sl]['model_attn'], src_test, trg_test)

# ============================================================
# 5. Attentionマトリクスの可視化
# ============================================================

## Attention重みを「見る」

Attentionの最大の利点の一つは**解釈可能性**です。

Attention重み行列を可視化すると：
- 行 = Decoderのタイムステップ（出力系列の各位置）
- 列 = Encoderのタイムステップ（入力系列の各位置）
- セルの色 = その入力位置への注目度（明るいほど高い）

### 反転タスクでの期待パターン

系列反転タスクでは、**反対角線（anti-diagonal）パターン** が期待されます。
- 出力の最初のトークン → 入力の最後のトークンに注目
- 出力の最後のトークン → 入力の最初のトークンに注目

これが実際に学習されるか確認しましょう！

In [ ]:
# ============================================================
# Attention ヒートマップ可視化関数
# ============================================================

def plot_attention(src_tokens, trg_tokens, attention_weights, ax=None, title='Attention重み行列'):
    """Attention重み行列をヒートマップで可視化
    
    Parameters
    ----------
    src_tokens : list  入力トークンのリスト
    trg_tokens : list  出力トークンのリスト
    attention_weights : (trg_len, src_len) のnumpy配列
    ax : matplotlib.axes (optional)
    title : str
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 8))
    
    im = ax.imshow(attention_weights, cmap='YlOrRd', aspect='auto', vmin=0, vmax=1)
    
    ax.set_xticks(range(len(src_tokens)))
    ax.set_xticklabels([str(t) for t in src_tokens], fontsize=12)
    ax.set_yticks(range(len(trg_tokens)))
    ax.set_yticklabels([str(t) for t in trg_tokens], fontsize=12)
    
    ax.set_xlabel('入力系列（Encoder）', fontsize=12)
    ax.set_ylabel('出力系列（Decoder）', fontsize=12)
    ax.set_title(title, fontsize=13, fontweight='bold')
    
    # 各セルに数値を表示
    for i in range(len(trg_tokens)):
        for j in range(len(src_tokens)):
            val = attention_weights[i, j]
            color = 'white' if val > 0.5 else 'black'
            ax.text(j, i, f'{val:.2f}', ha='center', va='center', 
                    fontsize=9, color=color)
    
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    return ax

print("✅ Attention可視化関数定義完了")

In [ ]:
# ============================================================
# 複数例のAttentionヒートマップ
# ============================================================

# 系列長6のモデルで可視化（見やすいため）
test_sl = 6
model_attn_6 = results[test_sl]['model_attn']
src_6, trg_6 = datasets[test_sl]

model_attn_6.eval()
with torch.no_grad():
    output_6, attn_6 = model_attn_6(src_6[:4], trg_6[:4], teacher_forcing_ratio=0.0)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for idx in range(3):
    src_tokens = src_6[idx].tolist()
    trg_tokens = trg_6[idx].tolist()
    pred_tokens = output_6[idx].argmax(dim=1).tolist()
    attn_matrix = attn_6[idx].numpy()  # (trg_len, src_len)
    
    correct = pred_tokens == trg_tokens
    status = "正解" if correct else "不正解"
    
    plot_attention(
        src_tokens, trg_tokens, attn_matrix, ax=axes[idx],
        title=f'例{idx+1} ({status})\n入力: {src_tokens}\n予測: {pred_tokens}'
    )

plt.suptitle('Attention重み行列の可視化（系列反転タスク）\n期待: 反対角線パターン', 
             fontsize=15, fontweight='bold', y=1.08)
plt.tight_layout()
plt.show()

print("💡 反対角線パターンが見えれば、Attentionが正しく学習できています！")
print("   出力の1番目 → 入力の最後に注目、出力の最後 → 入力の1番目に注目")

In [ ]:
# ============================================================
# 系列長による Attention パターンの違い
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for idx, seq_len in enumerate(SEQ_LENGTHS):
    model_a = results[seq_len]['model_attn']
    src_d, trg_d = datasets[seq_len]
    
    model_a.eval()
    with torch.no_grad():
        out, aw = model_a(src_d[:1], trg_d[:1], teacher_forcing_ratio=0.0)
    
    src_tokens = src_d[0].tolist()
    trg_tokens = trg_d[0].tolist()
    attn_matrix = aw[0].numpy()
    
    im = axes[idx].imshow(attn_matrix, cmap='YlOrRd', aspect='auto', vmin=0, vmax=1)
    axes[idx].set_title(f'系列長 {seq_len}', fontsize=13, fontweight='bold')
    axes[idx].set_xlabel('入力位置')
    axes[idx].set_ylabel('出力位置')
    plt.colorbar(im, ax=axes[idx], fraction=0.046, pad=0.04)

plt.suptitle('系列長によるAttentionパターンの比較', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("📊 系列が長くなっても、Attentionは適切な位置に注目できています")

# ============================================================
# 6. Scaled Dot-Product Attention
# ============================================================

## Bahdanau → Luong → Scaled Dot-Product

Attention のスコア関数には複数のバリエーションがあります。

### Attention スコア関数の比較

| 名称 | 数式 | 特徴 |
|------|------|------|
| **Bahdanau (加算的)** | $v^T \tanh(W_s s_t + W_h h_i)$ | 最初の提案、パラメータが多い |
| **Luong (乗法的)** | $s_t^T h_i$ | シンプル、計算が速い |
| **Scaled Dot-Product** | $\frac{Q \cdot K^T}{\sqrt{d_k}}$ | Transformerで使用 |

### Q, K, V (Query, Key, Value) の定式化

Scaled Dot-Product Attentionでは、Attentionを**検索**の比喩で理解します：

- **Query (Q)**: 「何を探しているか」（Decoder状態に相当）
- **Key (K)**: 「何が利用可能か」のインデックス（Encoder状態に相当）
- **Value (V)**: 実際に取り出す情報（Encoder状態に相当）

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

### なぜ $\sqrt{d_k}$ でスケーリング？

次元数 $d_k$ が大きくなると：
1. ドット積の値が大きくなる（分散が $d_k$ に比例）
2. Softmaxの入力が極端に大きくなる
3. Softmaxの出力が one-hot に近づく（勾配消失）

$\sqrt{d_k}$ で割ることで、分散を1に保ちます。

In [ ]:
# ============================================================
# Scaled Dot-Product Attention 実装
# ============================================================

def scaled_dot_product_attention(Q, K, V, mask=None):
    """Scaled Dot-Product Attention
    
    Attention(Q, K, V) = softmax(Q K^T / sqrt(d_k)) V
    
    Parameters
    ----------
    Q : (batch, n_queries, d_k)  Query
    K : (batch, n_keys, d_k)     Key
    V : (batch, n_keys, d_v)     Value
    mask : (batch, n_queries, n_keys) or None  マスク（0の位置を無視）
    
    Returns
    -------
    output : (batch, n_queries, d_v)  Attention出力
    weights : (batch, n_queries, n_keys)  Attention重み
    """
    d_k = Q.shape[-1]
    
    # スコア計算: Q K^T / sqrt(d_k)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(d_k)
    
    # マスク適用（オプション）
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    
    # Softmax で正規化
    weights = F.softmax(scores, dim=-1)
    
    # 重み付き和
    output = torch.matmul(weights, V)
    
    return output, weights


# --- 動作テスト ---
batch, n_q, n_k, d_k, d_v = 2, 4, 6, 8, 8
Q = torch.randn(batch, n_q, d_k)
K = torch.randn(batch, n_k, d_k)
V = torch.randn(batch, n_k, d_v)

output, weights = scaled_dot_product_attention(Q, K, V)
print(f"Q shape: {Q.shape}")
print(f"K shape: {K.shape}")
print(f"V shape: {V.shape}")
print(f"Output shape: {output.shape}")
print(f"Weights shape: {weights.shape}")
print(f"Weights sum (per query): {weights[0].sum(dim=-1)}")
print("\n✅ Scaled Dot-Product Attention 動作確認完了")

In [ ]:
# ============================================================
# スケーリングの効果を実験的に確認
# ============================================================

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

dimensions = [4, 32, 256]
torch.manual_seed(42)

for idx, d in enumerate(dimensions):
    # ランダムなQ, K
    Q_test = torch.randn(1, 1, d)
    K_test = torch.randn(1, 10, d)
    V_test = torch.randn(1, 10, d)
    
    # スケーリングなし
    scores_no_scale = torch.matmul(Q_test, K_test.transpose(-2, -1))
    weights_no_scale = F.softmax(scores_no_scale, dim=-1)
    
    # スケーリングあり
    scores_scaled = scores_no_scale / np.sqrt(d)
    weights_scaled = F.softmax(scores_scaled, dim=-1)
    
    # スケーリングなし
    ax = axes[0, idx]
    bars = ax.bar(range(10), weights_no_scale[0, 0].detach().numpy(), 
                  color='#e74c3c', alpha=0.8)
    ax.set_title(f'd_k = {d}\nスケーリングなし', fontsize=12, fontweight='bold')
    ax.set_ylabel('Attention重み')
    ax.set_ylim(0, 1.05)
    ax.set_xlabel('Key位置')
    max_w = weights_no_scale[0, 0].max().item()
    ax.text(0.98, 0.95, f'max={max_w:.3f}', transform=ax.transAxes, 
            ha='right', va='top', fontsize=10,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    # スケーリングあり
    ax = axes[1, idx]
    bars = ax.bar(range(10), weights_scaled[0, 0].detach().numpy(),
                  color='#3498db', alpha=0.8)
    ax.set_title(f'd_k = {d}\nスケーリングあり (÷√{d})', fontsize=12, fontweight='bold')
    ax.set_ylabel('Attention重み')
    ax.set_ylim(0, 1.05)
    ax.set_xlabel('Key位置')
    max_w = weights_scaled[0, 0].max().item()
    ax.text(0.98, 0.95, f'max={max_w:.3f}', transform=ax.transAxes,
            ha='right', va='top', fontsize=10,
            bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))

plt.suptitle('Scaled Dot-Product: スケーリングの効果\n次元が大きくなるとスケーリングなしではone-hotに近づく', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("📊 次元が大きくなるほど:")
print("   - スケーリングなし → Softmaxが尖る（one-hot化）→ 勾配消失のリスク")
print("   - スケーリングあり → 滑らかな分布を維持 → 安定した学習")

# ============================================================
# 7. Self-Attentionへの布石
# ============================================================

## Cross-Attention から Self-Attention へ

ここまで学んだAttentionは **Cross-Attention**（交差注意）と呼ばれます：
- **Q** はDecoder（出力側）から来る
- **K, V** はEncoder（入力側）から来る
- 異なる2つの系列を「橋渡し」する

### もし Q, K, V が全部同じ系列だったら？

これが **Self-Attention**（自己注意）です！

$$\text{Self-Attention}(X) = \text{softmax}\left(\frac{X \cdot X^T}{\sqrt{d_k}}\right) X$$

- 同じ系列内で、各位置が**他のすべての位置**に注目
- 「この単語は、文中の他のどの単語と関連が強いか？」を学習
- RNNのように逐次的に処理する必要がない → **並列計算が可能**

### Transformer への道

**"Attention Is All You Need"** (Vaswani et al., 2017) の革命的な問いかけ：

> 「RNNを完全に取り除き、Self-Attentionだけでモデルを構築したらどうなるか？」

答え：**Transformer** — 現代のAI（GPT, BERT, etc.）の基盤アーキテクチャ

### 接続マップ

```
Unit 8 (本ユニット)             次のステップ
──────────────────        ────────────────────
RNN → LSTM/GRU             
  → Seq2Seq                 
    → Attention ─────────→ Unit 9: 時空間モデリング
      → Self-Attention ──→ Unit 12: Transformerアーキテクチャ
                           Unit 12: 完全な Transformer 実装
```

In [ ]:
# ============================================================
# Self-Attention デモ
# ============================================================

def self_attention_demo(X):
    """Self-Attention: Q = K = V = X
    
    Parameters
    ----------
    X : (batch, seq_len, d_model)
    
    Returns
    -------
    output : (batch, seq_len, d_model)
    weights : (batch, seq_len, seq_len)
    """
    return scaled_dot_product_attention(X, X, X)


# --- Self-Attention の動作を可視化 ---
torch.manual_seed(42)

# 簡単な例: 5トークンの系列
seq_len, d_model = 5, 8

# 意図的にいくつかのトークンを類似させる
X = torch.randn(1, seq_len, d_model)
# トークン0とトークン3を類似させる
X[0, 3] = X[0, 0] + torch.randn(d_model) * 0.1
# トークン1とトークン4を類似させる  
X[0, 4] = X[0, 1] + torch.randn(d_model) * 0.1

output, self_attn_weights = self_attention_demo(X)

print(f"入力 X shape: {X.shape}")
print(f"出力 shape: {output.shape}")
print(f"Self-Attention weights shape: {self_attn_weights.shape}")
print(f"\nSelf-Attention 重み行列:")
print(self_attn_weights[0].detach().numpy().round(3))

# --- 可視化 ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Self-Attention 重み
token_labels = ['Tok 0', 'Tok 1', 'Tok 2', 'Tok 3\n(≈Tok0)', 'Tok 4\n(≈Tok1)']
im = axes[0].imshow(self_attn_weights[0].detach().numpy(), cmap='YlOrRd', vmin=0, vmax=0.6)
axes[0].set_xticks(range(seq_len))
axes[0].set_xticklabels(token_labels, fontsize=10)
axes[0].set_yticks(range(seq_len))
axes[0].set_yticklabels(token_labels, fontsize=10)
axes[0].set_title('Self-Attention 重み行列', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Key（参照先）')
axes[0].set_ylabel('Query（注目元）')
for i in range(seq_len):
    for j in range(seq_len):
        val = self_attn_weights[0, i, j].item()
        color = 'white' if val > 0.35 else 'black'
        axes[0].text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=9, color=color)
plt.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)

# Cross-Attention vs Self-Attention の概念図
ax = axes[1]
ax.axis('off')
ax.set_xlim(0, 10)
ax.set_ylim(0, 8)

# Cross-Attention
ax.text(2.5, 7.5, 'Cross-Attention', fontsize=13, fontweight='bold', ha='center',
        bbox=dict(boxstyle='round', facecolor='#3498db', alpha=0.3))
ax.text(2.5, 6.5, 'Q ← Decoder', fontsize=11, ha='center')
ax.text(2.5, 5.8, 'K, V ← Encoder', fontsize=11, ha='center')
ax.text(2.5, 5.0, '異なる2つの系列間', fontsize=10, ha='center', style='italic')

# Arrow
ax.annotate('', xy=(6.5, 5.5), xytext=(4.5, 5.5),
            arrowprops=dict(arrowstyle='->', lw=3, color='#e74c3c'))
ax.text(5.5, 6.0, '発展', fontsize=11, ha='center', color='#e74c3c', fontweight='bold')

# Self-Attention
ax.text(7.5, 7.5, 'Self-Attention', fontsize=13, fontweight='bold', ha='center',
        bbox=dict(boxstyle='round', facecolor='#2ecc71', alpha=0.3))
ax.text(7.5, 6.5, 'Q = K = V = X', fontsize=11, ha='center')
ax.text(7.5, 5.8, '（同じ系列から）', fontsize=11, ha='center')
ax.text(7.5, 5.0, '系列内の関係を捉える', fontsize=10, ha='center', style='italic')

# Timeline
ax.text(5, 3.5, 'Attentionの進化', fontsize=13, fontweight='bold', ha='center')
timeline_y = 2.5
points = [
    (1, 'Bahdanau\n(2015)'),
    (3.5, 'Luong\n(2015)'),
    (6, 'Scaled\nDot-Product'),
    (8.5, 'Transformer\n(2017)'),
]
ax.plot([1, 8.5], [timeline_y, timeline_y], 'k-', lw=2)
for x, label in points:
    ax.plot(x, timeline_y, 'ko', markersize=8)
    ax.text(x, timeline_y - 0.7, label, ha='center', va='top', fontsize=9)

plt.suptitle('Cross-Attention → Self-Attention → Transformer', 
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("💡 ポイント:")
print("   - Tok 0 と Tok 3 は類似 → 互いに高いAttention重み")
print("   - Tok 1 と Tok 4 は類似 → 互いに高いAttention重み")
print("   - Self-Attentionは系列内の類似パターンを自動的に発見！")

In [ ]:
# ============================================================
# Self-Attention: 実用的なデモ（学習可能なQ,K,V変換付き）
# ============================================================

class SelfAttentionLayer(nn.Module):
    """学習可能なSelf-Attention層
    
    入力Xから線形変換でQ, K, Vを生成し、
    Scaled Dot-Product Attentionを適用する。
    これがTransformerの基本ビルディングブロック。
    """
    def __init__(self, d_model):
        super().__init__()
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
    
    def forward(self, X, mask=None):
        Q = self.W_q(X)
        K = self.W_k(X)
        V = self.W_v(X)
        output, weights = scaled_dot_product_attention(Q, K, V, mask)
        return output, weights


# --- デモ ---
torch.manual_seed(42)
d_model = 16
seq_len = 6
batch = 1

self_attn_layer = SelfAttentionLayer(d_model)
X = torch.randn(batch, seq_len, d_model)

output, weights = self_attn_layer(X)
print(f"入力 shape: {X.shape}")
print(f"出力 shape: {output.shape}")
print(f"Attention weights shape: {weights.shape}")
print(f"\n✅ 学習可能なSelf-Attention層の動作確認完了")
print(f"\nこのSelf-Attention層がTransformerの核心部品です。")
print(f"Transformerは、この層を複数スタックし、Multi-Head化したものです。")

# ============================================================
# 8. Unit 8 総まとめ
# ============================================================

## Unit 8「シーケンスモデリング」完了！

7つのノートブックを通じて、系列データの処理方法を体系的に学びました。

| NB | テーマ | キーポイント |
|---|--------|-------------|
| 120 | シーケンスとは | 静的vs系列、MLPの限界、状態の概念 |
| 121 | バニラRNN | 状態方程式、重み共有、forward実装 |
| 122 | BPTT | 時間方向の逆伝播、勾配消失/爆発、Adding Problem |
| 123 | LSTM | 4ゲート、セル状態の勾配高速道路 |
| 124 | GRU | 2ゲート簡略化、三者比較、時系列予測 |
| 125 | Seq2Seq | Encoder-Decoder、コンテキストベクトル、Teacher Forcing |
| **126** | **Attention** | **Bahdanau Attention、Scaled Dot-Product、Self-Attention** |

### 習得したスキル

- **理論**: 系列モデリングの基本原理（状態、再帰、ゲート機構）
- **実装**: RNN/LSTM/GRU/Seq2Seq/Attention をゼロから実装
- **デバッグ**: 勾配消失/爆発の診断と対策
- **評価**: 訓練曲線、Attention可視化による解釈

### 次のステップ

```
Unit 8 完了
    ├── Unit 9:  時空間モデリング（空間 + 時間の統合）
    ├── Unit 12: 言語モデリング（完全なTransformer実装）
    └── Unit 13: 表現学習（埋め込みと事前学習）
```

# ============================================================
# よくある間違い・チートシート
# ============================================================

## よくある間違い

### 1. Attention重みの次元ミス
```python
# ❌ 間違い: softmaxの軸が違う
weights = F.softmax(scores, dim=0)  # バッチ方向に正規化してしまう

# ✅ 正解: Key（入力）方向に正規化
weights = F.softmax(scores, dim=-1)  # 最後の次元（Key方向）
```

### 2. スケーリングの忘れ
```python
# ❌ 間違い: d_kが大きいときに勾配消失
scores = Q @ K.T

# ✅ 正解: sqrt(d_k) でスケーリング
scores = Q @ K.T / np.sqrt(d_k)
```

### 3. コンテキストベクトルの計算ミス
```python
# ❌ 間違い: 重みとValueの次元が合わない
context = weights * V  # element-wise になってしまう

# ✅ 正解: 行列積で重み付き和
context = torch.bmm(weights.unsqueeze(1), V).squeeze(1)
```

## チートシート

| 概念 | 数式 | PyTorch |
|------|------|--------|
| Bahdanau Score | $v^T \tanh(W_s s + W_h h)$ | `v(tanh(Ws(s) + Wh(h)))` |
| Dot-Product Score | $s^T h$ | `torch.matmul(s, h.T)` |
| Scaled Score | $\frac{QK^T}{\sqrt{d_k}}$ | `torch.matmul(Q, K.T) / sqrt(d_k)` |
| Attention重み | $\text{softmax}(\text{score})$ | `F.softmax(scores, dim=-1)` |
| コンテキスト | $\sum \alpha_i v_i$ | `torch.bmm(weights, V)` |
| Self-Attention | $Q=K=V=X$ | `sdp_attention(X, X, X)` |

# ============================================================
# 学習チェックリスト
# ============================================================

## 学習目標の達成確認

以下の項目について、自信を持って説明できるか確認してください。

- [ ] **Bahdanau Attention** の数式を書き下し、各パラメータの役割を説明できる
- [ ] **Attention重み行列** の読み方を理解し、モデルの振る舞いを解釈できる
- [ ] **Scaled Dot-Product Attention** で $\sqrt{d_k}$ スケーリングが必要な理由を説明できる
- [ ] **Q, K, V** の概念と、検索の比喩で直感的に説明できる
- [ ] **Cross-Attention** と **Self-Attention** の違いを明確に区別できる
- [ ] Attentionが **Seq2Seqのボトルネック問題** をどう解決するか説明できる
- [ ] Self-Attentionから **Transformer** への接続を概念的に理解している

# ============================================================
# 9. 自己評価クイズ
# ============================================================

## 理解度チェック

---

### Q1: なぜAttentionはボトルネック問題を解決できるのか？

<details>
<summary>回答を表示</summary>

従来のSeq2Seqでは、入力系列全体を1つの固定長ベクトル（コンテキストベクトル）に圧縮するため、入力が長くなると情報が失われました。

Attentionでは、Decoderの各ステップで**すべてのEncoder隠れ状態**を参照し、必要な情報を動的に選択します。これにより、入力のどの部分の情報も直接アクセス可能になり、ボトルネックが解消されます。
</details>

---

### Q2: コピータスク（入力をそのまま出力）のAttention行列はどんなパターンになる？

<details>
<summary>回答を表示</summary>

**対角線（diagonal）パターン**になります。

- 出力の1番目 → 入力の1番目に注目
- 出力の2番目 → 入力の2番目に注目
- ...

反転タスクの場合は**反対角線（anti-diagonal）パターン**になります。
</details>

---

### Q3: Scaled Dot-Product Attentionで $\sqrt{d_k}$ で割る理由は？

<details>
<summary>回答を表示</summary>

次元数 $d_k$ が大きくなると、ドット積の分散が $d_k$ に比例して大きくなります。

分散が大きい = 値が極端に大きくなる → Softmaxの出力がone-hotに近づく → 勾配が極端に小さくなる（勾配消失）。

$\sqrt{d_k}$ で割ることで分散を1に正規化し、Softmaxが滑らかな分布を出力できるようにします。
</details>

---

### Q4: Cross-AttentionとSelf-Attentionの違いは？

<details>
<summary>回答を表示</summary>

**Cross-Attention**: Q と K,V が**異なる系列**から来る。
- 例: Q=Decoder隠れ状態, K,V=Encoder隠れ状態
- 2つの系列間の対応関係を学習

**Self-Attention**: Q, K, V が**同じ系列**から来る（Q=K=V=X）。
- 系列内の各位置が、他のすべての位置との関係を学習
- RNNなしで長距離依存を捉えられる
</details>

---

### Q5: 「Attention Is All You Need」でRNNを取り除けた理由は？

<details>
<summary>回答を表示</summary>

RNNの主な役割は「系列の各位置間の依存関係を捉えること」でした。

Self-Attentionは、各位置が**直接すべての他の位置を参照**できるため、RNNのように逐次的に情報を伝播させる必要がありません。

さらに、Self-Attentionには以下の利点があります：
1. **並列計算が可能**（RNNは逐次的で遅い）
2. **長距離依存を直接捉えられる**（RNNは距離に応じて情報が減衰）
3. **位置エンコーディング**を追加することで順序情報も保持

これらの利点により、RNNをSelf-Attentionで完全に置き換えたTransformerが誕生しました。
</details>

# ============================================================
# 参考文献
# ============================================================

## References

1. **Bahdanau, D., Cho, K., & Bengio, Y.** (2015). "Neural Machine Translation by Jointly Learning to Align and Translate." *ICLR 2015*. https://arxiv.org/abs/1409.0473

2. **Luong, M.-T., Pham, H., & Manning, C. D.** (2015). "Effective Approaches to Attention-based Neural Machine Translation." *EMNLP 2015*. https://arxiv.org/abs/1508.04025

3. **Vaswani, A., et al.** (2017). "Attention Is All You Need." *NeurIPS 2017*. https://arxiv.org/abs/1706.03762

---

*このノートブックは Machine Learning Playground の一部です。*  
*Unit 8「シーケンスモデリング」— NB126: Attention ― Transformerへの架け橋*